# Project 1: Object Detection on Traffic Light Dataset

#### Maximum Points: 100

In this project, the goal is to train an object detection model to detect traffic light color using the dataset provided in this notebook. You can use any framework or model, for example YOLO or RT-DETR and inference techniques like SAHI for better detection of small objects. The focus is on creating an accurate and efficient solution for identifying traffic light color.

**The dataset is provided in YOLO format. However, you are free to convert it to any format of your choice according to the library that you are using**

**The dataset contains a total of 5 classes:**

```python
0: 'Green',
1: 'off',
2: 'red',
3: 'wait_on',
4: 'yellow'  
```

## Your Tasks

* Train an object detection model.
* Perform inference on the given video file.
* Share your inference video in an accessible format, such as uploading it to YouTube or providing a shared drive link.
* Provide a video or writeup link explaining your code and approach.

**Alternatively, you can also upload your writeup document to the online lab.**

**Mark Distribution**:

<div align="center">
    <table border="1" style="border-collapse: collapse;">
        <tr>
            <td><h3>No.</h3></td>
            <td><h3>Criteria</h3></td>
            <td><h3>Marks</h3></td>
        </tr>
        <tr>
            <td><h3>1.</h3></td>
            <td><h3> Dataset Visualization</h3></td>
            <td><h3>15</h3></td>
        </tr>
        <tr>
            <td><h3>2.</h3></td>
            <td><h3>mAP50-95 >= 48%</h3></td>
            <td><h3>70</h3></td>
        </tr>
        <tr>
            <td><h3>3.</h3></td>
            <td><h3>Inference on Test Video</h3></td>
            <td><h3>10</h3></td>
        </tr>
        <tr>
            <td><h3>4.</h3></td>
            <td><h3>Writeup/Video Explanation</h3></td>
            <td><h3>5</h3></td>
        </tr>
    </table>
</div>



---
<h2 style = "color: green;">Dataset Download</h2>

The Traffic Light dataset description is given in the README.dataset.txt file inside the Small Traffic Light dataset folder. You can download the zip file of the dataset from the link provided below:

[Traffic Light Dataset](https://www.dropbox.com/scl/fi/v60994ucoillzfzcwe1kl/Small-Traffic-Light.v1i.yolov11.zip?rlkey=b28kal2b8egup5u73vigh953f&st=kqiijqnv&dl=1)

---

**The notebook is divided into multiple grading sections. You have to write code, as mentioned for each section.  For other helper functions, you can write `.py` files and import them in the notebook. You have to submit the notebook along with `.py` files. Your submitted code must be runnable without any bug.**

<h2 style = "color: green">1. Visualize dataset [Points 15]</h2>


**In this sub-section, you have to plot a few images with their bounding box labels overlayed similar to one shown below.**

<img src = "https://www.dropbox.com/scl/fi/iw5c2fcib8cqufnskpfz3/media_dataset_visualization.png?rlkey=ccsl1ntwwh331j3bixnwilgsy&st=hzmq47bb&dl=1" width = "1000" height = "550">


In [ ]:
# For colab environment
# !pip install ultralytics torchinfo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import cv2
import wandb
import time
from ultralytics import YOLO
from pathlib import Path
import shutil
import os

In [ ]:
# For colab environment
# from google.colab import drive
# drive.mount('/content/drive')
#
# wandb.login()

In [ ]:
from config import Config
from env import local_env, colab_env
from find_off_labels import find_all_off_images

In [ ]:
# config = Config(colab_env, training=False)
config = Config(local_env, training=False)

In [ ]:
def plot_images_with_bounding_boxes(
        images,
        pred_bounding_boxes=None,
        pred_bounding_boxes_class_labels=None,
        pred_boxes_confidence=None,
        true_bounding_boxes=None,
        true_bounding_boxes_class_labels=None,
        white_text_background=True,
):
    for i, image in enumerate(images):
        h, w = image.shape[:2]
        fig, ax = plt.subplots(1, figsize=(14, 10))
        ax.imshow(image)
        ax.axis('off')

        if true_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(true_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='orange', facecolor='none')
                ax.add_patch(rect)
                if true_bounding_boxes_class_labels is not None:
                    label = true_bounding_boxes_class_labels[i][j]
                    if white_text_background:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left')

        if pred_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(pred_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='blue', facecolor='none')
                ax.add_patch(rect)
                if pred_boxes_confidence is not None:
                    conf_str = f"{int(pred_boxes_confidence[i][j] * 100)}%"
                    if pred_bounding_boxes_class_labels is not None:
                        label = f"{pred_bounding_boxes_class_labels[i][j]} {conf_str}"
                    else:
                        label = conf_str
                    if white_text_background:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left')

        plt.tight_layout()
        plt.show()

In [ ]:
def plot_images_with_ground_truth_bounding_boxes(
    class_labels,
    image_paths,
    image_labels,
    white_text_background=True,
):
    images = []
    all_boxes = []
    all_class_labels = []
    for image_path, image_label in zip(image_paths, image_labels):
        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        images.append(image)
        boxes = []
        box_labels = []
        with open(image_label, 'r') as label_file:
            for line in label_file:
                line_segments = line.strip().split()
                box_labels.append(class_labels[int(line_segments[0])])
                coords = [float(c) for c in line_segments[-4:]]
                boxes.append(coords)
        all_boxes.append(boxes)
        all_class_labels.append(box_labels)
    plot_images_with_bounding_boxes(images, true_bounding_boxes=all_boxes, true_bounding_boxes_class_labels=all_class_labels, white_text_background=white_text_background)

In [ ]:
def plot_val_images_with_ground_truth_bounding_boxes(
    num_images_to_show=3,
    white_text_background=True,
):
    ids = np.random.choice(config.env.fetch_original_val_ids(), num_images_to_show)
    image_paths = []
    image_labels = []
    for image_id in ids:
        image_paths.append(f'{config.env.fixed_val_images_folder}{image_id}.jpg')
        image_labels.append(f'{config.env.fixed_val_labels_folder}{image_id}.txt')
    plot_images_with_ground_truth_bounding_boxes(config.class_labels, image_paths, image_labels, white_text_background=white_text_background)

### Correcting mislabeled data

There are only 4 'off' labeled traffic lights in the dataset, and only one is labeled correctly. So, after posting in the course forums and getting instructor approval, I am removing the 'off' class and only training my model to predict the other 4 classes. I fixed the incorrectly labeled 'off' boxes in the dataset as well. Shown below are the 4 images that originally included a traffic light labeled 'off' - 2 in the train set and 2 in the validation set. As you can see, only the 3rd image actually has a traffic light that is off; the other 3 are mislabeled.

In [ ]:
off_image_paths, off_image_labels = find_all_off_images(config.env.original_dataset_root_folder)
plot_images_with_ground_truth_bounding_boxes(['green', 'off', 'red', 'wait_on', 'yellow'], off_image_paths, off_image_labels, white_text_background=True)

To correct for the issue, I adjusted the labels files to move all 'yellow' labeled lights (formerly index 4) to index 1. In the above images with 'off' labels, I removed the label corresponding to the off light in the 3rd image, and changed the class of the other 3 incorrect labels to the correct class. You can find the full updated dataset at this google drive link:

https://drive.google.com/drive/folders/1Poob3f5RCuguhmDyR_Auay33OjPUkZE7?usp=sharing

Below, I print out a random sampling of a few of the validation images, as requested:

In [ ]:
plot_val_images_with_ground_truth_bounding_boxes(num_images_to_show=3)

<h2 style = "color: green;">2. Train the Model </h2>

In [ ]:
def train_yolo():
    start_time = time.time()
    print(f't=0: Initializing training setup...')

    wandb.init(
        project='traffic-light-detection',
        name=f'run={int(start_time)}',
        config=config.named_dict(),
    )

    model = YOLO(config.model_name)

    def on_train_epoch_end(trainer):
        epoch = trainer.epoch + 1
        loss_items = trainer.label_loss_items(trainer.tloss, prefix='train')
        metrics = {}
        for key in ('train/box_loss', 'train/cls_loss', 'train/dfl_loss'):
            if key in loss_items:
                metrics[key] = loss_items[key]
        for key in ('lr/pg0', 'lr/pg1', 'lr/pg2'):
            if key in trainer.lr:
                metrics[key] = trainer.lr[key]
        if metrics:
            wandb.log(metrics, step=epoch)

    def on_fit_epoch_end(trainer):
        epoch = trainer.epoch + 1
        metrics = {}
        for key in ('metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss'):
            if key in trainer.metrics:
                metrics[key] = trainer.metrics[key]
        if metrics:
            wandb.log(metrics, step=epoch)

    model.add_callback('on_train_epoch_end', on_train_epoch_end)
    model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

    results = model.train(
        data=config.env.fixed_yolo_config_yaml,
        epochs=config.max_epochs,
        patience=config.patience,
        imgsz=1280,
        rect=True,
        batch=24,
        optimizer='AdamW',
        cos_lr=False,
        seed=config.seed,
        amp=config.use_amp,
        workers=config.num_workers,
        device=config.env.device,
        verbose=True,
        plots=True,
        lr0 = 0.001,
        lrf = 0.2,
        warmup_epochs = 3.0,
        warmup_momentum = 0.8,
        warmup_bias_lr = 0.1,
        augmentations=config.train_transforms,
        hsv_h=config.hsv_h,
        hsv_s=config.hsv_s,
        hsv_v=config.hsv_v,
    )

    best_src = Path(results.save_dir) / 'weights' / 'best.pt'
    shutil.copy(str(best_src), config.env.saved_weights_filepath)
    print(f'Best model saved to {config.env.saved_weights_filepath}')

    artifact = wandb.Artifact(name='best-model', type='model')
    artifact.add_file(config.env.saved_weights_filepath)
    wandb.log_artifact(artifact)

    wandb.finish()
    return results

### WandB logs:

https://wandb.ai/kyledunne-n-a/traffic-light-detection/runs/x5hvqs8d?nw=nwuserkyledunne

<h2 style = "color: green;">3. Validate the Model </h2>

In [ ]:
def validate_yolo():
    model = YOLO(config.env.saved_weights_filepath)
    val_results = model.val(
        data=config.env.fixed_yolo_config_yaml,
        imgsz=1280,
        batch=8,
        workers=config.num_workers,
        device=config.env.device,
        verbose=True,
        plots=True,
        save_json=True,
    )
    return val_results

In [ ]:
validate_yolo()

<h2 style = "color: green;">4. Inference on given video [10 Points]</h2>

<h3 style><a href = "https://www.dropbox.com/scl/fi/v3uwaid41m5jypk8mhplr/inference_traffic_light_video.mp4?rlkey=u4l9kxa0av81qxcfod06z7e0g&st=ab83a04b&dl=1">Inference Video</a></h3>


<video
     controls
     src="https://www.dropbox.com/scl/fi/v3uwaid41m5jypk8mhplr/inference_traffic_light_video.mp4?rlkey=u4l9kxa0av81qxcfod06z7e0g&st=kp0dxjz3&dl=1"
     width="640">
 </video>

In [ ]:
def predict_video():
    model = YOLO(config.env.saved_weights_filepath)

    video = cv2.VideoCapture(config.env.video_input_filepath)
    if not video.isOpened():
        print('Error opening video file')
        return

    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = video.get(cv2.CAP_PROP_FPS)

    out_dir = os.path.dirname(config.env.video_output_filepath)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    output_file = cv2.VideoWriter(
        filename=config.env.video_output_filepath,
        fourcc=cv2.VideoWriter_fourcc(*'mp4v'),
        fps=float(fps),
        frameSize=(width, height),
        isColor=True,
    )

    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break

        results = model.predict(
            source=frame,
            imgsz=1280,
            device=config.env.device,
            verbose=False,
        )

        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()  # [N, 4] pixel coords (x1, y1, x2, y2)
            confs = result.boxes.conf.cpu().numpy()  # [N]
            cls_ids = result.boxes.cls.cpu().numpy()  # [N]
            for (x1, y1, x2, y2), conf, cls_id in zip(boxes, confs, cls_ids):
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color=(0, 165, 255), thickness=2)
                label = f'{config.class_labels[int(cls_id)]} {int(conf * 100)}%'
                cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX,
                            fontScale=0.6, color=(0, 165, 255), thickness=2)

        output_file.write(frame)

    video.release()
    output_file.release()
    print(f'Video saved to: {config.env.video_output_filepath}')

In [ ]:
predict_video()

<h2 style = "color: green;">5. Writeup/Video Explanation [5 Points]</h2>

* Give a detailed written description or a video/screen recording explaining your approach and code.
* Once completed upload the notebook to the online lab and submit.
* Provide the link for the explanation and inferenced video in this section.

### Writeup:

For this traffic light object detection project, I trained a YOLO 26 model, specifically the x-p2 variant - x is for extra large, meaning the most parameters, and the p2 variant is optimized for better performance on detecting small objects. This was crucial for this project, as many of the traffic lights in the provided dataset are very small in comparison to the images, which are 1920x1080.

This large resolution presented a challenge. Originally, my plan was to use SAHI (slicing-aided hyper-inference) through the `sahi` library. For each image in the training and validation dataset, I created 8 640x640 overlapping slices of the image, and trained a YOLO model on the 640x640 slices, using SAHI to stitch the predictions back together for inference. This allowed the model to train on subsections of the image at the original resolution, meaning there was no downscaling (which seemed important given how small many of the traffic lights were), while still fitting the model into a fairly small amount of VRAM even with a large batch size. However, performance with this pipeline, while not terrible, was not as good as I had hoped. I am sure there were more optimizations I could have made to improve the performance while keeping the same approach, but I was worried that the model was suffering due to only seeing small portions of the overall image. Also, with each image sliced into 8 subsections, many of the individual sections (>65%) had no traffic lights at all, so a large portion of the dataset was not really helping at all with teaching the model what traffic lights look like. To curb this, I made a change where I removed about 2/3rds of the empty slices; this means there were still some slices with no traffic lights, but they didn't overwhelm the positive examples. This change improved performance a small but significant amount.

In the middle of this process, while looking over the dataset and training performance, I noticed a potential issue with the original dataset. There are 5 classes our model is supposed to predict: `['green', 'off', 'red', 'wait_on', 'yellow']`. While there are hundreds of examples of the other kinds of lights in the training and validation data, I noticed that there were extremely few examples of `'off'` traffic lights. In fact, only 2 images in the original train set and original val set have an off traffic light. My model was performing extremely poorly on specifically the `'off'` class metrics, and it's not hard to understand why, when there are only 2 images for it to learn from in the training set. To try to fix this, I made 10 duplicates of the 2 images in the train set with 'off' class traffic lights, and added them to the dataset - essentially just telling ultralytics to train on those images 11 times each epoch instead of once (and they each will have slightly different random augmentations applied). However, even after this change, performance on that class was still terrible. To try to figure out why, I made visualizations of all 4 of the images with an 'off' label, and found that they were actually mislabeled. 3 of the 4 'off' lights were actually red. No wonder my model was doing so poorly on that class! This was important reminder to delve deeply into the dataset before trying to run any models or make any architecture decisions; catching this earlier would have saved me a lot of time. I posted on the course forums about this issue and the instructor agreed that it is appropriate to ignore the 'off' class and only try to predict the other 4 classes. I reworked the original dataset to accomplish this, by updating the label files to change the class index of 'yellow' traffic lights to 1 (formerly 'off') instead of 4. For the affected images that were incorrectly labeled, I changed the mislabeled lights to the correct label (and remove the bounding box label for the one instance of an actually correctly labeled 'off' light).

I made the updated dataset publicly available on Google Drive at this link: https://drive.google.com/drive/folders/1Poob3f5RCuguhmDyR_Auay33OjPUkZE7?usp=sharing

Once I had updated the dataset, I decided to try a simplified approach. I used google colab's newly available G4 GPU instance - which is very fast and 96 GB of VRAM - to run a YOLO 26 model on the full images, rather than small images slices like before. I set the image size to 1280, with rect=True, which downscales the long side of the image (1920) down to 1280 and the short side of the image proportionally. This is a moderate downscale (~33%), but not huge, so the model was still able to pick up the small traffic lights pretty well. I also switched to YOLO's P2 variant, which is designed with extra elements dedicated to local features, and is supposed to perform better for small object detection. With the YOLO26 X-P2 variant, I was able to train with a batch size of 24 and use about 80 GB of the 96 GB of VRAM available on the G4 colab instance. This model performed well, achieving a mAP50-95 of approximately .545 on the validation set, well above the required .48 for this project. I think that this could be improved significantly with further hyperparameter optimization. In addition, with access to more VRAM, I would recommend training at a higher resolution, such as 1536 or even ideally the original 1920, to further boost performance, so long as it can be done without pushing the batch size too low. Ultimately, the original SAHI approach proved to be unnecessary for this project, though if my computing resources were more constrained, I think it likely would have been a strong choice to achieve comparable performance on a lower VRAM budget.

### Inference Video YouTube Link:

(todo)